# Stage 3 — Report Generator Training (Google Colab)

**What this notebook does:**  
Trains `HybridReportGenerator` (flan-t5-large on T4/A100, flan-t5-base on free T4)  
using pre-cached features from Stage 1 & 2. All checkpoints auto-save to Google Drive.

**Before running:**
- Upload your project zip to Google Drive → `MyDrive/sae-frag.zip`
- Upload the pre-built cache files to Google Drive:
  - `MyDrive/sae_frag_store/cache_train.pt`
  - `MyDrive/sae_frag_store/cache_val.pt`
- Set runtime to **T4 GPU**: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1: Mount Drive & unpack project ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE       = '/content/drive/MyDrive'
REPO_DIR    = '/content/sae-frag'          # extracted here (fast local SSD)
STORE_DIR   = f'{REPO_DIR}/store'
CKPT_DIR    = f'{REPO_DIR}/checkpoints/stage3'
DRIVE_CKPT  = f'{DRIVE}/sae_frag_checkpoints'   # where we sync back to Drive

# Unzip project from Drive to local /content (much faster I/O than Drive)
if not os.path.isdir(REPO_DIR):
    print('Extracting project...')
    shutil.unpack_archive(f'{DRIVE}/sae-frag.zip', '/content/')

os.makedirs(STORE_DIR,  exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)
os.makedirs(DRIVE_CKPT, exist_ok=True)

print(f'Repo   : {REPO_DIR}')
print(f'Store  : {STORE_DIR}')
print(f'Ckpts  : {CKPT_DIR}')
print('Done.')

In [ ]:
# ── Cell 2: Copy cache files from Drive → local SSD ──────────────────────
# Only copies if not already present (re-run safe)
import shutil, os

for fname in ['cache_train.pt', 'cache_val.pt']:
    src = f'{DRIVE}/sae_frag_store/{fname}'
    dst = f'{STORE_DIR}/{fname}'
    if not os.path.exists(dst):
        print(f'Copying {fname} ...')
        shutil.copy2(src, dst)
        print(f'  Done ({os.path.getsize(dst)/1e6:.0f} MB)')
    else:
        print(f'{fname} already present ({os.path.getsize(dst)/1e6:.0f} MB)')

# Copy any existing stage3 checkpoints from Drive (for resume)
for fname in ['best_generator.pth', 'last_generator.pth', 'resume.pt']:
    src = f'{DRIVE_CKPT}/{fname}'
    dst = f'{CKPT_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'Restoring checkpoint {fname} ...')
        shutil.copy2(src, dst)

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────
!pip install -q transformers sentencepiece rouge-score
print('Dependencies ready.')

In [ ]:
# ── Cell 4: GPU check ────────────────────────────────────────────────────
import torch, os, sys

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.chdir(REPO_DIR)

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 14:
    print('\n>> VRAM >= 14 GB: switching to flan-t5-LARGE for best results')
    USE_LARGE = True
else:
    print('\n>> VRAM < 14 GB: using flan-t5-base')
    USE_LARGE = False

In [ ]:
# ── Cell 5: Configure model size based on available VRAM ─────────────────
hybrid_gen_path = os.path.join(REPO_DIR, 'src', 'rag', 'hybrid_generator.py')

with open(hybrid_gen_path) as f:
    code = f.read()

if USE_LARGE:
    code = code.replace('model_name="google/flan-t5-base"',
                        'model_name="google/flan-t5-large"')
    BATCH_SIZE  = 4
    ACCUM_STEPS = 4
    LR_T5       = 5e-6
    LR_NEW      = 2e-5
else:
    code = code.replace('model_name="google/flan-t5-large"',
                        'model_name="google/flan-t5-base"')
    BATCH_SIZE  = 8
    ACCUM_STEPS = 2
    LR_T5       = 1e-5
    LR_NEW      = 5e-5

with open(hybrid_gen_path, 'w') as f:
    f.write(code)

print(f'Model : {"flan-t5-large" if USE_LARGE else "flan-t5-base"}')
print(f'Batch : {BATCH_SIZE} x accum {ACCUM_STEPS} = effective {BATCH_SIZE*ACCUM_STEPS}')

In [ ]:
# ── Cell 6: Full Stage 3 training loop ───────────────────────────────────
import random, sys, os
import torch
import torch.amp
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from transformers import get_cosine_schedule_with_warmup

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from rag.hybrid_generator import HybridReportGenerator

# ── Config ────────────────────────────────────────────────────────
DEVICE       = torch.device('cuda')
NUM_EPOCHS   = 100
WARMUP_STEPS = 300 if USE_LARGE else 200
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 0.5
PATIENCE     = 5
TRAIN_CACHE  = f'{STORE_DIR}/cache_train.pt'
VAL_CACHE    = f'{STORE_DIR}/cache_val.pt'
BEST_CKPT    = f'{CKPT_DIR}/best_generator.pth'
LAST_CKPT    = f'{CKPT_DIR}/last_generator.pth'
RESUME_FILE  = f'{CKPT_DIR}/resume.pt'

# ── Dataset ───────────────────────────────────────────────────────
class CachedFeaturesDataset(Dataset):
    def __init__(self, path):
        print(f'Loading {path} ...')
        data = torch.load(path, weights_only=False)
        self.aligned  = [[v['aligned_features'] for v in i['variants']] for i in data]
        self.entities = [[v['entity_vector']    for v in i['variants']] for i in data]
        self.reps     = [[v['retrieved_text']   for v in i['variants']] for i in data]
        self.targets  = [i['target'] for i in data]
        del data
        print(f'  {len(self.targets)} samples loaded.')
    def __len__(self): return len(self.targets)
    def __getitem__(self, idx):
        k = random.randrange(len(self.aligned[idx]))
        return self.aligned[idx][k], self.entities[idx][k], self.reps[idx][k], self.targets[idx]

def collate_fn(batch):
    afs, evs, reps, tgts = zip(*batch)
    return torch.stack(afs), torch.stack(evs), list(reps), list(tgts)

train_ds = CachedFeaturesDataset(TRAIN_CACHE)
val_ds   = CachedFeaturesDataset(VAL_CACHE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=collate_fn, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1,          shuffle=False, num_workers=2, collate_fn=collate_fn)

# ── Model ─────────────────────────────────────────────────────────
generator = HybridReportGenerator().to(DEVICE)
generator.t5.gradient_checkpointing_enable()   # saves ~30% VRAM

_new = {'visual_proj','visual_norm','visual_drop','entity_embed'}
t5_params  = [p for n,p in generator.named_parameters() if not any(k in n for k in _new) and p.requires_grad]
new_params = [p for n,p in generator.named_parameters() if any(k in n for k in _new) and p.requires_grad]
optimizer  = torch.optim.AdamW([{'params':t5_params,'lr':LR_T5},{'params':new_params,'lr':LR_NEW}], weight_decay=WEIGHT_DECAY)
total_steps = (len(train_loader) // ACCUM_STEPS) * NUM_EPOCHS
scheduler   = get_cosine_schedule_with_warmup(optimizer, WARMUP_STEPS, total_steps)

# ── Resume ────────────────────────────────────────────────────────
start_epoch = 0; best_val_loss = float('inf'); no_improve = 0
if os.path.exists(RESUME_FILE):
    r = torch.load(RESUME_FILE, map_location=DEVICE, weights_only=False)
    compat = all(generator.state_dict()[k].shape == v.shape for k,v in r['model'].items() if k in generator.state_dict())
    if compat:
        generator.load_state_dict(r['model'], strict=False)
        optimizer.load_state_dict(r['optimizer'])
        scheduler.load_state_dict(r['scheduler'])
        start_epoch = r['epoch'] + 1; best_val_loss = r['best_val_loss']; no_improve = r.get('no_improve', 0)
        print(f'Resumed from epoch {r["epoch"]+1} | val={best_val_loss:.4f}')
    else:
        print('Checkpoint architecture mismatch — starting fresh.')
else:
    print('No checkpoint — starting fresh.')

print(f'\nTraining: {len(train_ds)} train / {len(val_ds)} val')
print(f'Epochs: {NUM_EPOCHS} (from {start_epoch+1}) | Steps: {total_steps}\n')

# ── Train loop ────────────────────────────────────────────────────
for epoch in range(start_epoch, NUM_EPOCHS):
    generator.train(); total_loss = 0.0; optimizer.zero_grad()
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')

    for step, (af, ev, reps, reports) in enumerate(loop):
        af = af.to(DEVICE); ev = ev.to(DEVICE)
        prompts = ['Generate a detailed radiology report for the chest X-ray.'] * len(reports)
        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            loss = generator(region_features=af, entity_vector=ev, retrieved_texts=reps, prompt_texts=prompts, target_texts=reports)
            loss = loss / ACCUM_STEPS
        loss.backward()
        total_loss += loss.item() * ACCUM_STEPS
        if (step+1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(generator.parameters(), GRAD_CLIP)
            optimizer.step(); optimizer.zero_grad(); scheduler.step()
        loop.set_postfix(loss=f'{total_loss/(step+1):.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    avg_train = total_loss / len(train_loader)

    generator.eval(); val_loss = 0.0
    with torch.no_grad():
        for vaf, vev, vreps, vreps_tgt in tqdm(val_loader, desc='  Val', leave=False):
            vaf = vaf.to(DEVICE); vev = vev.to(DEVICE)
            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                val_loss += generator(region_features=vaf, entity_vector=vev, retrieved_texts=list(vreps),
                                      prompt_texts=['Generate a detailed radiology report for the chest X-ray.'],
                                      target_texts=list(vreps_tgt)).item()
    avg_val = val_loss / max(len(val_loader), 1)
    torch.cuda.empty_cache()
    print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} | train={avg_train:.4f} | val={avg_val:.4f}')

    torch.save(generator.state_dict(), LAST_CKPT)
    torch.save({'epoch':epoch,'model':generator.state_dict(),'optimizer':optimizer.state_dict(),
                'scheduler':scheduler.state_dict(),'best_val_loss':best_val_loss,'no_improve':no_improve}, RESUME_FILE)

    if avg_val < best_val_loss:
        best_val_loss = avg_val; no_improve = 0
        torch.save(generator.state_dict(), BEST_CKPT)
        print(f'  ** Best saved (val={best_val_loss:.4f})')
    else:
        no_improve += 1
        print(f'  No improvement {no_improve}/{PATIENCE}')
        if no_improve >= PATIENCE:
            print('Early stopping.'); break

print(f'\nDone. Best val={best_val_loss:.4f}')

In [ ]:
# ── Cell 7: Save checkpoints back to Drive ───────────────────────────────
# Run this after training (or any time) to back up to Drive
import shutil, os

os.makedirs(DRIVE_CKPT, exist_ok=True)
for fname in ['best_generator.pth', 'last_generator.pth', 'resume.pt']:
    src = f'{CKPT_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_CKPT}/{fname}')
        print(f'Saved {fname} -> Drive')
print('All checkpoints backed up to Google Drive.')